## Scraping AWOIAF Characters — v3 (include-list: History + Recent Events only)

v2 used an **exclude-list** (`affiliated = everything inside mw-parser-output except the Family section`). That still let navboxes, succession templates, and 'Members of X' tables leak in — that's why characters like *Demon of Darry* ended up affiliated to 140+ characters spanning every Hand of the King, Master of Coin, and Kingsguard knight in history.

v3 inverts the policy. `parse_affiliated_v3` collects links **only from inside `History` and `Recent Events` h2 sections** — the narrative-prose sections. Everything else — infobox, navboxes, succession boxes, References, See also, Notes, Behind the Scenes — is ignored.

Infobox fields (father, mother, spouse, lover, issue, allegiance) are still captured by `parse_infobox` into their own columns — that side table is intact.

Output: `characters_enriched_v3.csv`.

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor


### Setup
Same UA, session, and `INFOBOX_LABELS` as v1/v2.

In [2]:
BASE = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

INFOBOX_LABELS = {
    'Father': 'father', 'Fathers': 'father',
    'Mother': 'mother', 'Mothers': 'mother',
    'Spouse': 'spouse', 'Spouses': 'spouse',
    'Lover': 'lover', 'Lovers': 'lover', 'Paramour': 'lover',
    'Issue': 'issue', 'Issues': 'issue',
    'Allegiance': 'allegiance', 'Allegiances': 'allegiance',
}

session = requests.Session()
session.headers.update(headers)


### Helpers (unchanged from v2)

In [3]:
def slug_to_url(slug):
    return BASE + PREFIX + quote(slug, safe="_/(),'")


def href_to_slug(href):
    if not href.startswith(PREFIX):
        return None
    if 'redlink=1' in href or 'action=' in href:
        return None
    slug = href[len(PREFIX):].split('#', 1)[0]
    if not slug or slug.startswith('Special:'):
        return None
    return unquote(slug)


def cell_links_or_text(td):
    slugs = []
    seen = set()
    for a in td.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    if slugs:
        return ';'.join(slugs)
    text = td.get_text(' ', strip=True)
    return text if text else ''


def parse_infobox(soup):
    data = {col: '' for col in INFOBOX_LABELS.values()}
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        return data
    for row in infobox.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        label = th.get_text(strip=True).rstrip(':')
        col = INFOBOX_LABELS.get(label)
        if col is None:
            continue
        if data[col] == '':
            data[col] = cell_links_or_text(td)
    return data


### NEW `parse_affiliated_v3` — include-list

Builds a set of every DOM node that lives inside an `<h2>History</h2>` or `<h2>Recent Events</h2>` section: the h2 heading itself, every sibling until the next h2, and all their descendants. Then only collects `<a>` tags whose `id()` is in that set.

Matching is **case-insensitive** — the survey of section headings showed `Recent Events` is overwhelmingly canonical but `Recent events` (lowercase 'e') appears on 19 pages, and a single `Histoy` typo exists. We catch all of them by lowercasing the heading and matching against `INCLUDED_H2_SECTIONS`.

Adding more sections later is a one-line edit — e.g. include `'appearances'` if you decide that section is signal.

In [4]:
INCLUDED_H2_SECTIONS = {'history', 'recent events', 'recent event', 'histoy'}  # lowercased; 'histoy' = wiki typo


def parse_affiliated_v3(soup, self_slug):
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ''

    included = set()
    for h in content.find_all('h2'):
        span = h.find('span', class_='mw-headline')
        if not span:
            continue
        if span.get_text(strip=True).lower() not in INCLUDED_H2_SECTIONS:
            continue
        included.add(id(h))
        for d in h.find_all(True):
            included.add(id(d))
        for sib in h.find_next_siblings():
            if sib.name == 'h2':
                break
            included.add(id(sib))
            for d in sib.find_all(True):
                included.add(id(d))

    slugs = []
    seen = set()
    for a in content.find_all('a'):
        if id(a) not in included:
            continue
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug == self_slug or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    return ';'.join(slugs)


### Per-character scrape
Identical shape to v2's `scrape_character` — only the affiliated parser is swapped.

In [5]:
EMPTY = {'father': '', 'mother': '', 'spouse': '', 'lover': '', 'issue': '',
         'allegiance': '', 'affiliated': ''}


def scrape_character(row):
    name, slug = row['name'], row['ID']
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {slug}: {e}')
        return {'name': name, 'ID': slug, **EMPTY}
    if resp.status_code != 200:
        print(f'  ! {resp.status_code} for {slug}')
        return {'name': name, 'ID': slug, **EMPTY}
    soup = BeautifulSoup(resp.content, 'html.parser')
    info = parse_infobox(soup)
    return {
        'name': name,
        'ID': slug,
        'father': info['father'],
        'mother': info['mother'],
        'spouse': info['spouse'],
        'lover': info['lover'],
        'issue': info['issue'],
        'allegiance': info['allegiance'],
        'affiliated': parse_affiliated_v3(soup, slug),
    }


### Run the scrape
~3,689 character pages, 8 workers, checkpoints every 100 rows. Same runtime as v1/v2 (≈ 10 min).

In [6]:
characters = pd.read_csv('../csvs/characters.csv').to_dict('records')
print(f'{len(characters)} characters to enrich')

COLUMNS = ['name', 'ID', 'father', 'mother', 'spouse', 'lover', 'issue', 'allegiance', 'affiliated']
OUT = '../csvs/characters_enriched_v3.csv'
WORKERS = 8

rows = []
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for i, row in enumerate(executor.map(scrape_character, characters), 1):
        rows.append(row)
        if i % 50 == 0:
            print(f'  {i}/{len(characters)}')
        if i % 100 == 0:
            pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'Saved {len(rows)} rows to {OUT}')


3690 characters to enrich
  50/3690
  100/3690
  150/3690
  200/3690
  250/3690
  300/3690
  350/3690
  400/3690
  450/3690
  500/3690
  550/3690
  600/3690
  650/3690
  700/3690
  750/3690
  800/3690
  850/3690
  900/3690
  950/3690
  1000/3690
  1050/3690
  1100/3690
  1150/3690
  1200/3690
  1250/3690
  1300/3690
  1350/3690
  1400/3690
  1450/3690
  1500/3690
  1550/3690
  1600/3690
  1650/3690
  1700/3690
  1750/3690
  1800/3690
  1850/3690
  1900/3690
  1950/3690
  2000/3690
  2050/3690
  2100/3690
  2150/3690
  2200/3690
  2250/3690
  2300/3690
  2350/3690
  2400/3690
  2450/3690
  2500/3690
  2550/3690
  2600/3690
  2650/3690
  2700/3690
  2750/3690
  2800/3690
  2850/3690
  2900/3690
  2950/3690
  3000/3690
  3050/3690
  3100/3690
  3150/3690
  3200/3690
  3250/3690
  3300/3690
  3350/3690
  3400/3690
  3450/3690
  3500/3690
  3550/3690
  3600/3690
  3650/3690
Saved 3690 rows to characters_enriched_v3.csv


### Restrict slug-valued columns to characters only
Identical to v2 section 3. Drops non-character internal links from `father, mother, spouse, lover, issue, affiliated`; drops free-text fallbacks from `allegiance`.

In [7]:
df = pd.read_csv(OUT).fillna('')
character_ids = set(df['ID'])

SLUG_COLUMNS = ['father', 'mother', 'spouse', 'lover', 'issue', 'affiliated']


def keep_characters(cell):
    if not cell:
        return ''
    return ';'.join(s for s in cell.split(';') if s in character_ids)


def count_entries(col):
    return df[col].str.count(';').add(df[col].ne('').astype(int)).sum()


for col in SLUG_COLUMNS:
    before = count_entries(col)
    df[col] = df[col].map(keep_characters)
    after = count_entries(col)
    print(f'  {col:11s}: {before:>6} -> {after:<6}  (dropped {before - after})')

before = count_entries('allegiance')
df['allegiance'] = df['allegiance'].map(
    lambda c: ';'.join(s for s in c.split(';') if s and ' ' not in s and '[' not in s) if c else ''
)
after = count_entries('allegiance')
print(f'  {"allegiance":11s}: {before:>6} -> {after:<6}  (dropped {before - after} free-text entries)')

df.to_csv(OUT, index=False)


  father     :    995 -> 914     (dropped 81)
  mother     :    515 -> 468     (dropped 47)
  spouse     :    670 -> 520     (dropped 150)
  lover      :    204 -> 188     (dropped 16)
  issue      :   1611 -> 1396    (dropped 215)
  affiliated :  60686 -> 23322   (dropped 37364)
  allegiance :   4111 -> 4103    (dropped 8 free-text entries)


### Diff vs. v1 and v2
Per-character count of affiliated links across the three versions. v3 should be much tighter than v2 — and the gap from v2 → v3 is roughly *navbox + succession + 'Members of X' leakage*.

Spot-check the top-15 character pairs by v2→v3 drop: expect to see Kingsguard, Hands of the King, Masters of Coin/Ships/Laws, and Grand Maesters dominate the list — those are the characters whose pages historically had the heaviest navboxes.

In [8]:
v1 = pd.read_csv('../csvs/characters_enriched.csv').fillna('')
v2 = pd.read_csv('characters_enriched_v2.csv').fillna('')
v3 = pd.read_csv(OUT).fillna('')


def count_links(s):
    return len([x for x in s.split(';') if x])


merged = (v1[['ID', 'affiliated']].rename(columns={'affiliated': 'aff_v1'})
          .merge(v2[['ID', 'affiliated']].rename(columns={'affiliated': 'aff_v2'}), on='ID')
          .merge(v3[['ID', 'affiliated']].rename(columns={'affiliated': 'aff_v3'}), on='ID'))

merged['n_v1'] = merged['aff_v1'].map(count_links)
merged['n_v2'] = merged['aff_v2'].map(count_links)
merged['n_v3'] = merged['aff_v3'].map(count_links)
merged['v1_v2_drop'] = merged['n_v1'] - merged['n_v2']
merged['v2_v3_drop'] = merged['n_v2'] - merged['n_v3']

print(f'Total affiliated links:  v1={merged["n_v1"].sum():,}   v2={merged["n_v2"].sum():,}   v3={merged["n_v3"].sum():,}')
print(f'Mean per character:      v1={merged["n_v1"].mean():.1f}   v2={merged["n_v2"].mean():.1f}   v3={merged["n_v3"].mean():.1f}')
print(f'Characters with 0 v3 links: {(merged["n_v3"] == 0).sum()} / {len(merged)}')
print()
print('Top 15 characters by v2 -> v3 drop (navbox/succession noise removed):')
print(merged.nlargest(15, 'v2_v3_drop')[['ID', 'n_v1', 'n_v2', 'n_v3', 'v2_v3_drop']].to_string(index=False))


Total affiliated links:  v1=56,089   v2=42,421   v3=23,322
Mean per character:      v1=15.2   v2=11.5   v3=6.3
Characters with 0 v3 links: 667 / 3690

Top 15 characters by v2 -> v3 drop (navbox/succession noise removed):
                   ID  n_v1  n_v2  n_v3  v2_v3_drop
       Demon_of_Darry   145   145     2         143
       Robert_Flowers   143   143     0         143
      Alyn_Connington   143   143     2         141
              Greydon   139   139     0         139
Lucan_(Grand_Maester)   143   143     4         139
              Malleon   141   141     2         139
                Clegg   141   141     4         137
           Aethelmure   140   140     5         135
      Barristan_Selmy   181   181    70         111
 Viserys_II_Targaryen   156   132    33          99
      Corlys_Velaryon   157   152    54          98
     Tyrion_Lannister   196   184    86          98
         Ryam_Redwyne   126   125    32          93
         Eddard_Stark   195   177    87          90

### Sanity: Demon of Darry
The pretext for v3. Should drop dramatically from the v2 number of ~140.

In [9]:
row = v3[v3['ID'] == 'Demon_of_Darry']
if len(row):
    aff = row['affiliated'].iloc[0]
    print(f'Demon_of_Darry v3 affiliated count: {count_links(aff)}')
    print(f'Affiliated: {aff[:500]}{"..." if len(aff) > 500 else ""}')
else:
    print('Demon_of_Darry not found in v3 output')


Demon_of_Darry v3 affiliated count: 2
Affiliated: Jaime_Lannister;Loras_Tyrell
